# Derivatives

## What's covered

- The **derivative** formally — slope of a tangent line, written as a limit
- **Notation** — `f'(x)`, `df/dx`, Leibniz vs Lagrange vs Newton
- **Basic rules** — power rule, linearity (sum, constant multiple)
- **Product rule** and **quotient rule**
- A first look at the **chain rule** (notebook 4 takes it apart in full)
- Derivatives of the staple functions — `e^x`, `log x`, `sin x`, `cos x`
- Derivatives of every ML activation you've heard of — sigmoid, tanh, ReLU, softplus
- **Finite differences** — forward, backward, central; how to verify any derivative numerically


## The derivative, formally

Notebook 1 ended with the picture. The derivative of `f` at `x` is the slope of the tangent line to `f` at the point `(x, f(x))`, defined as the limit of secant slopes:

$$
f'(x) = \lim_{h \to 0} \frac{f(x + h) - f(x)}{h}
$$

Two things to keep straight:

- **`f'(x)` is a number.** It tells you the slope at *one specific point*. Plug in `x = 2`, you get one number.
- **`f'` is a function.** As `x` varies, the slope varies. So `f'` is itself a new function — the **derivative function** — whose value at every `x` is the slope of the original `f` at that `x`.

In ML, we usually want both readings at once. We treat `f'` as the function (so we can evaluate it at any current parameter setting), and we evaluate it at the specific parameter values we currently have.

**Geometric check.** If `f` is sloping up at `x`, then `f'(x) > 0`. Sloping down ⇒ `f'(x) < 0`. Horizontal (flat) ⇒ `f'(x) = 0`. That last one is the entire game of optimization — flat points are candidates for minima.

**Existence.** Not every continuous function is differentiable. `|x|` and `ReLU(x)` are continuous everywhere but have no derivative at `x = 0` — the left-side slope (`-1`, `0` respectively) and right-side slope (`+1`, `+1`) disagree. The limit doesn't exist there. Frameworks paper over this with subgradients, as we mentioned in notebook 1.


## Notation

Three notations coexist. Use whichever clarifies — they all mean the same thing.

| Notation | Style | Best for |
|---|---|---|
| `f'(x)` | Lagrange | Concise; when the variable is obvious |
| `df/dx` | Leibniz | When you want to spotlight *which variable* is changing; chain rule reads naturally |
| `Df(x)` or `D_x f` | Operator | Theoretical writing, multi-input/output |
| `ẋ` | Newton | Physics, time derivatives — rare in ML |

A few more useful pieces:

- **Higher-order derivatives** — `f''(x)` or `d²f/dx²` for the second derivative. The Hessian (notebook 5) is the multivariable version.
- **Evaluation at a point** — `f'(a)` or `df/dx |_{x=a}`. The bar with subscript pins down "evaluated here."
- **The differential** — `dy = f'(x) \, dx`. Reads as "a small change in `y` is approximately `f'(x)` times a small change in `x`." This is the linear approximation, the seed of Taylor expansion (notebook 6).


## Basic rules — power and linearity

A handful of rules let us differentiate almost any function ML uses. The first three are the absolute essentials.

**Power rule.** For any real `n`:

$$
\frac{d}{dx} x^n = n \, x^{n-1}
$$

Examples: `d/dx [x^2] = 2x`, `d/dx [x^{1/2}] = (1/2) x^{-1/2}`, `d/dx [1/x] = d/dx [x^{-1}] = -x^{-2}`.

**Constant rule.** The derivative of a constant is zero. If `c` doesn't depend on `x`, `dc/dx = 0`. Loss functions never lose constant terms during gradient descent — they just don't matter.

**Linearity.** Derivatives respect scaling and addition:

$$
\frac{d}{dx} [c \, f(x)] = c \, f'(x), \qquad \frac{d}{dx} [f(x) + g(x)] = f'(x) + g'(x)
$$

Combined: the derivative of any linear combination is the same linear combination of derivatives. Mean squared error is built from this — `MSE(w) = (1/n) Σ (y_i - hat{y}_i)^2` differentiates term by term.


In [ ]:
import numpy as np

def finite_diff(f, x, h=1e-5):
    """Central difference — the most accurate single-step numerical derivative."""
    return (f(x + h) - f(x - h)) / (2 * h)

# Power rule: d/dx [x^3] = 3x^2
f = lambda x: x ** 3
df_analytic = lambda x: 3 * x ** 2

for x in [-2.0, 0.5, 1.0, 3.0]:
    print(f"x = {x:>4}   analytic f'(x) = {df_analytic(x):>8.4f}   numerical = {finite_diff(f, x):>8.4f}")

# Linearity: d/dx [3 x^2 - 5x + 7] = 6x - 5
g = lambda x: 3 * x ** 2 - 5 * x + 7
dg_analytic = lambda x: 6 * x - 5
print()
for x in [-1.0, 0.0, 2.0]:
    print(f"x = {x:>4}   analytic g'(x) = {dg_analytic(x):>8.4f}   numerical = {finite_diff(g, x):>8.4f}")


## Product and quotient rules

Differentiating a product is **not** the product of derivatives. It's:

$$
(fg)'(x) = f'(x) g(x) + f(x) g'(x)
$$

Memory aid: *"derivative of first times second, plus first times derivative of second."*

Quotient rule:

$$
\left(\frac{f}{g}\right)'(x) = \frac{f'(x) g(x) - f(x) g'(x)}{g(x)^2}
$$

The quotient rule looks heavy but you almost never need to memorize it — you can usually rewrite `f/g` as `f \cdot g^{-1}` and use product rule plus chain rule. In ML you'll meet the quotient rule mostly in derivations of softmax and the sigmoid derivative (both follow from `1/(1+e^{-x})`-type expressions).

**A reusable trick.** When `f = u/v` and `v > 0`, taking logs gives `\log f = \log u - \log v`. Differentiating with the chain rule and rearranging is sometimes cleaner than the quotient rule directly — this is called **logarithmic differentiation**. It's how you'd derive the gradient of the softmax cleanly.


In [ ]:
# Product rule check: d/dx [(x^2)(sin x)] = 2x sin x + x^2 cos x
f = lambda x: (x ** 2) * np.sin(x)
df = lambda x: 2 * x * np.sin(x) + (x ** 2) * np.cos(x)

print("Product rule on x^2 * sin(x):")
for x in [0.5, 1.0, 2.0]:
    print(f"  x={x}  analytic={df(x):.6f}  numerical={finite_diff(f, x):.6f}")

# Quotient rule check: d/dx [sin x / x] = (cos x . x - sin x) / x^2
g = lambda x: np.sin(x) / x
dg = lambda x: (np.cos(x) * x - np.sin(x)) / (x ** 2)

print("\nQuotient rule on sin(x)/x:")
for x in [0.5, 1.0, 2.0]:
    print(f"  x={x}  analytic={dg(x):.6f}  numerical={finite_diff(g, x):.6f}")


## A first taste of the chain rule

The chain rule is the heart of backpropagation, and notebook 4 unpacks it carefully. For now, a one-line version:

$$
\frac{d}{dx} f(g(x)) = f'(g(x)) \cdot g'(x)
$$

In words: *outside derivative evaluated at the inside, times derivative of the inside.* If you set `u = g(x)` and `y = f(u)`, Leibniz notation makes it pop:

$$
\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}
$$

The `du` "cancels" — not formally, but it's the right mnemonic. Derivatives chain by **multiplying**. Every gradient flowing through every layer of a neural network is one long product of small derivatives, and that product is exactly what the chain rule computes.

Example: `f(x) = sin(x^2)`. Set `u = x^2`, so `f = sin(u)`. Then `df/dx = cos(u) · 2x = 2x cos(x^2)`.


## Derivatives of the staple functions

These are the building blocks. You should know them cold.

| Function | Derivative |
|---|---|
| `c` (constant) | `0` |
| `x^n` | `n x^{n-1}` |
| `e^x` | `e^x` |
| `a^x` (`a > 0`) | `a^x \ln a` |
| `\ln x` (`x > 0`) | `1/x` |
| `\log_a x` | `1 / (x \ln a)` |
| `\sin x` | `\cos x` |
| `\cos x` | `-\sin x` |
| `\tan x` | `\sec^2 x` |

The two most ML-relevant ones are `e^x` (its own derivative — the only function with this property up to a constant) and `\ln x` (whose derivative is `1/x`, which is why log-likelihoods are so much easier to differentiate than likelihoods).


In [ ]:
# Verify the staples
checks = [
    ("e^x at x=1",        np.exp, lambda x: np.exp(x), 1.0),
    ("ln x at x=2",       np.log, lambda x: 1 / x,     2.0),
    ("sin x at x=pi/4",   np.sin, lambda x: np.cos(x), np.pi / 4),
    ("cos x at x=pi/3",   np.cos, lambda x: -np.sin(x), np.pi / 3),
]

print(f"{'identity':<22} {'analytic':>12} {'numerical':>14}")
for name, f, df, x in checks:
    print(f"{name:<22} {df(x):>12.6f} {finite_diff(f, x):>14.6f}")


## ML activations — derive every one

These are the derivatives you'll write on whiteboards. Each has a clean closed form, and several have a beautiful "you can express the derivative in terms of the original output" property that makes backprop cheap.

**Sigmoid.** `\sigma(x) = 1 / (1 + e^{-x})`. Quotient rule (or rewrite as `(1 + e^{-x})^{-1}` and chain rule):

$$
\sigma'(x) = \sigma(x) \, (1 - \sigma(x))
$$

That's the magic property — the derivative is computable from the **output** with one multiply and one subtract. During forward pass we already have `σ(x)`, so the backward pass is free.

**Tanh.** Closely related:

$$
\tanh'(x) = 1 - \tanh^2(x)
$$

Same trick — derivative in terms of output.

**ReLU.** `\text{ReLU}(x) = \max(0, x)`. Piecewise linear, so the derivative is piecewise constant:

$$
\text{ReLU}'(x) = \begin{cases} 1 & x > 0 \\ 0 & x < 0 \\ \text{undefined} & x = 0 \end{cases}
$$

Frameworks pick a subgradient at `x = 0` — usually `0` or `1`. In practice it never matters.

**Softplus.** `\text{softplus}(x) = \ln(1 + e^x)`. Beautiful:

$$
\text{softplus}'(x) = \sigma(x)
$$

The derivative of the smooth ReLU is the sigmoid. The shapes of the activation and the sigmoid line up perfectly.

**GELU and Swish/SiLU** have closed-form derivatives too but the formulas are longer. GELU's derivative involves the standard-normal density; Swish's is `σ(x) + x σ(x)(1-σ(x))`. Frameworks compute them efficiently — you won't be writing them by hand at the whiteboard.


In [ ]:
def sigmoid(x):  return 1.0 / (1.0 + np.exp(-x))
def tanh(x):     return np.tanh(x)
def relu(x):     return np.maximum(0.0, x)
def softplus(x): return np.log1p(np.exp(x))

# Analytical derivatives
def d_sigmoid(x):  s = sigmoid(x); return s * (1 - s)
def d_tanh(x):     return 1 - np.tanh(x) ** 2
def d_relu(x):     return (x > 0).astype(float)
def d_softplus(x): return sigmoid(x)  # the beautiful identity

print(f"{'fn':<10} {'x':>6} {'analytic':>14} {'numerical':>14}")
for x in [-2.0, -0.5, 0.5, 2.0]:
    for name, f, df in [("sigmoid", sigmoid, d_sigmoid),
                        ("tanh",    tanh,    d_tanh),
                        ("relu",    relu,    d_relu),
                        ("softplus",softplus,d_softplus)]:
        a, n = df(x), finite_diff(f, x)
        print(f"{name:<10} {x:>6} {a:>14.6f} {n:>14.6f}")
    print()


## Finite differences — verifying any derivative

Whenever you derive a gradient by hand, sanity-check it numerically. Three flavors, in increasing accuracy:

**Forward difference** — `(f(x + h) - f(x)) / h`. Error is `O(h)`. The most natural definition but the least accurate.

**Backward difference** — `(f(x) - f(x - h)) / h`. Same `O(h)` accuracy.

**Central difference** — `(f(x + h) - f(x - h)) / (2h)`. Error is `O(h^2)`. Costs one extra function evaluation but the accuracy jump is huge. **This is the one to use** for verification.

**Choosing `h`.** Too large and you lose accuracy to truncation error (the Taylor remainder); too small and floating-point round-off destroys you. The sweet spot for central differences in double precision is around `h ≈ 1e-5` to `1e-6`. For forward differences, `h ≈ 1e-8` is theoretically optimal but very brittle.

**The standard recipe for verifying a hand-derived gradient:** compute the analytical value, compute the central-difference value, check that the relative error is below `1e-5` or so. Every neural network library has a `gradcheck` function that does exactly this.


## Where this appears in ML

The single-variable derivative shows up the moment you have one scalar parameter. Most of ML has many parameters at once, but the *operations* on each weight are still single-variable derivatives — chained together.

- **Backpropagation.** Each layer's "local" gradient is a single-variable (or vector) derivative — `σ'`, `ReLU'`, `tanh'`. The chain rule glues them together.
- **Activation derivatives in code.** Sigmoid, tanh, softplus all use the "derivative-from-output" trick: store the forward output once, reuse for the backward pass. Saves a multiply per neuron per step.
- **Cross-entropy + softmax.** Their gradient simplifies to `(prediction − target)` because of how `d/dx \ln σ(x) = 1 - σ(x)` plays with the softmax's quotient structure. We derive it in notebook 5.
- **Logistic regression derivative.** `d/dw \ln σ(w · x) = (1 - σ(w · x)) x`. Same trick.
- **Gradient checking in PyTorch.** `torch.autograd.gradcheck` runs central differences against autograd's output during testing.
- **Learning-rate intuition.** A step of size `α` in parameter space changes the loss by approximately `-α |∇L|^2` (to first order). The "first order" there is the single-variable derivative applied in each parameter direction.
- **Smooth activations vs ReLU.** GELU, Swish, Mish were designed by playing with derivatives — wanting non-monotone behavior, wanting derivatives that vary smoothly through zero.

Next notebook: **partial derivatives and gradients** — we lift everything we just built from one variable to many, and the *gradient* shows up as the multivariable answer to "which direction is uphill?"
